# E4 — Mini-progetto finale (SOLUZIONI)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrealorenzon/SIMA2026/blob/soluzioni/E4_progetto/E4_soluzioni.ipynb)

## Il mini-progetto

Un lavoro **a casa, in gruppi da 3**, tra la L8 e l'ultimo incontro: rispondere a **una domanda** con i dati, usando tutto quello che avete visto nel corso.

| Quando | Cosa |
|---|---|
| prima della L8 | scegliete la domanda e i dati (compito della L7). Alla L8 li controlliamo insieme |
| dopo la L8 | lavorate sul notebook, seguendo le sezioni qui sotto |
| prima delle feste | sportello facoltativo su Zoom, per chi è bloccato |
| il giorno prima dell'ultimo incontro | consegnate il notebook: in Colab, **Condividi** → chiunque abbia il link, e mandate il link al docente |
| ultimo incontro | ogni gruppo presenta in **circa 10 minuti**: la domanda, un grafico, la risposta, i limiti. Poi qualche domanda dagli altri |

Alla presentazione **chiunque del gruppo** deve saper spiegare ogni passaggio: le domande possono essere per tutti.

**Dati possibili:** il dataset delle imprese del corso; i dati di [Our World in Data](https://github.com/owid/co2-data) visti nella L7 (o altri dataset di Our World in Data); un dataset pubblico che conosci per il tuo dottorato (Eurostat, ISTAT, Banca d'Italia, World Bank...).

**Usate Gemini quanto volete.** Ma ogni numero che presentate dovete saperlo spiegare e averlo verificato.

## Come viene valutato

| Criterio | Cosa guardiamo |
|---|---|
| Domanda | è chiara e si può davvero rispondere con i dati scelti? |
| Pulizia | i problemi dei dati sono stati trovati, e le decisioni scritte? |
| Verifiche | conteggi prima e dopo, valori mancanti, unioni controllate |
| Grafico | onesto: titolo, assi, unità, ordine |
| Risposta e limiti | la conclusione è proporzionata ai dati? quante osservazioni ci sono dietro? |
| Riproducibilità | Runtime → Riavvia ed esegui tutto funziona; c'è uno script `.py` finale |

> Quella che segue è **una** soluzione d'esempio, su una domanda scelta dal docente. Serve a mostrare la struttura, non è l'unica risposta possibile.

## 1. La domanda

Tra il 2000 e il 2022, i paesi che sono cresciuti di più hanno anche aumentato di più le emissioni di CO₂ pro capite?

## 2. I dati

Our World in Data, dataset su CO₂ ed energia, scaricato da GitHub a ottobre 2026. Una riga = un paese (o un aggregato!) in un anno.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

co2 = pd.read_csv("https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv")
print(co2.shape)

## 3. Pulizia

In [ ]:
paesi = co2[co2["iso_code"].notna()].copy()          # via gli aggregati (L7)
paesi = paesi[paesi["year"].isin([2000, 2022])]
paesi["pil_pc"] = paesi["gdp"] / paesi["population"]
w = paesi.pivot_table(index="country", columns="year", values=["pil_pc", "co2_per_capita"])
w.columns = [f"{v}_{a}" for v, a in w.columns]
print(len(w))
w = w.dropna()
print(len(w))

*Decisioni:* tolti gli aggregati (senza `iso_code`); tenuti solo i paesi con PIL ed emissioni in entrambi gli anni (si perdono i paesi senza dati sul PIL: il campione non è il mondo intero).

## 4. Analisi

In [ ]:
w["crescita_pil"] = (w["pil_pc_2022"] / w["pil_pc_2000"] - 1) * 100
w["crescita_co2"] = (w["co2_per_capita_2022"] / w["co2_per_capita_2000"] - 1) * 100
print(w[["crescita_pil", "crescita_co2"]].median().round(1))
m = smf.ols("crescita_co2 ~ crescita_pil", data=w).fit()
print(m.params.round(2), round(m.rsquared, 2), m.nobs)

## 5. Il grafico

In [ ]:
w.plot(kind="scatter", x="crescita_pil", y="crescita_co2", alpha=0.6)
plt.axhline(0, color="grey", lw=0.8)
plt.title("Crescita del PIL e delle emissioni pro capite, 2000-2022")
plt.xlabel("Crescita del PIL pro capite (%)")
plt.ylabel("Crescita della CO2 pro capite (%)")
plt.show()

## 6. La risposta e i limiti

Sì, in media: i paesi cresciuti di più hanno aumentato di più le emissioni pro capite (relazione positiva, vedi coefficiente). Ma la dispersione è grande: molti paesi sono cresciuti **riducendo** le emissioni (i punti sotto la linea dello zero). Limiti: è un'associazione tra paesi, non un effetto causale; mancano i paesi senza dati sul PIL; due soli anni possono dipendere da eventi particolari.

## 7. Lo script

In [ ]:
%%writefile progetto.py
"""Crescita del PIL e delle emissioni pro capite, 2000-2022 (dati Our World in Data)."""
import pandas as pd
import statsmodels.formula.api as smf

co2 = pd.read_csv("https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv")
paesi = co2[co2["iso_code"].notna() & co2["year"].isin([2000, 2022])].copy()
paesi["pil_pc"] = paesi["gdp"] / paesi["population"]
w = paesi.pivot_table(index="country", columns="year", values=["pil_pc", "co2_per_capita"]).dropna()
w.columns = [f"{v}_{a}" for v, a in w.columns]
w["crescita_pil"] = (w["pil_pc_2022"] / w["pil_pc_2000"] - 1) * 100
w["crescita_co2"] = (w["co2_per_capita_2022"] / w["co2_per_capita_2000"] - 1) * 100
m = smf.ols("crescita_co2 ~ crescita_pil", data=w).fit()
print(len(w), m.params.round(2).to_dict())


In [ ]:
!python progetto.py